# SenseVoice 混合噪音增强微调实验

**混合数据策略**：原始干净数据 + 噪音增强数据，1:1 混合

实验组：
- **混合A**: 原始数据 + 高斯合成噪音增强数据（1:1）
- **混合B**: 原始数据 + ESC-50 真实噪音增强数据（1:1）

## 1. 环境检查

In [ ]:
import torch
import subprocess
import json
import os
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

try:
    import funasr
    print(f"FunASR: {funasr.__version__}")
except ImportError:
    print("FunASR 未安装")

## 2. 数据准备

In [ ]:
import torch
import soundfile as sf
import json
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RAW_TRAIN = "/mnt/workspace/data/prepared_data/sensevoice/train.jsonl"
VAL_JSONL = "/mnt/workspace/data/prepared_data/sensevoice/val.jsonl"
PREPARED_DIR = "/mnt/workspace/data/prepared_data/sv_ablation"
ESC50_DIR = "/mnt/data/audio"
os.makedirs(PREPARED_DIR, exist_ok=True)

def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

raw_train = load_jsonl(RAW_TRAIN)
print(f"原始训练集: {len(raw_train)} 条")

# --- 高斯合成噪音增强 (A) ---
def add_gaussian_noise(audio_path, snr_db_range=(10, 20)):
    wav, sr = sf.read(audio_path)
    wav = torch.from_numpy(wav).float()
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = torch.randn_like(wav) * torch.sqrt(noise_power)
    return (wav + noise).numpy(), sr

def generate_noise_a_only(data, output_dir, output_filename="train_noise_a_only.jsonl"):
    """生成纯噪音A数据（不包含原始数据）"""
    noise_dir = os.path.join(output_dir, "noise_a_only")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)
    new_data = []
    for item in data:
        src = item['source']
        try:
            noisy_wav, sr = add_gaussian_noise(src)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_a.wav")
            sf.write(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")
    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  纯噪音A: {len(new_data)} 条")
    return jsonl_path

# --- ESC-50 真实噪音增强 (B) ---
ESC50_NOISES = [
    os.path.join(ESC50_DIR, f)
    for f in os.listdir(ESC50_DIR)
    if f.endswith('.wav')
]
print(f"  ESC-50 噪音文件: {len(ESC50_NOISES)} 条")

def add_real_noise(audio_path, noise_path, snr_db_range=(10, 20)):
    wav, sr = sf.read(audio_path)
    wav = torch.from_numpy(wav).float()
    noise, noise_sr = sf.read(noise_path)
    noise = torch.from_numpy(noise).float()
    if noise_sr != sr:
        new_len = int(noise.shape[0] * sr / noise_sr)
        noise = torch.nn.functional.interpolate(
            noise.view(1, 1, -1), size=new_len, mode='linear', align_corners=False
        ).squeeze()
    if noise.shape[0] < wav.shape[0]:
        pad = torch.zeros(wav.shape[0] - noise.shape[0])
        noise = torch.cat([noise, pad])
    else:
        noise = noise[:wav.shape[0]]
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = (noise ** 2).mean()
    adjusted_noise = noise * torch.sqrt(signal_power / (noise_power * (10 ** (snr_db / 10))))
    return (wav + adjusted_noise).numpy(), sr

def generate_noise_b_only(data, output_dir, output_filename="train_noise_b_only.jsonl"):
    """生成纯噪音B数据（不包含原始数据）"""
    if not ESC50_NOISES:
        print("  警告: 未找到 ESC-50 噪音文件")
        return None
    noise_dir = os.path.join(output_dir, "noise_b_only")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)
    new_data = []
    for item in data:
        src = item['source']
        noise_path = random.choice(ESC50_NOISES)
        try:
            noisy_wav, sr = add_real_noise(src, noise_path)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_b.wav")
            sf.write(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")
    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  纯噪音B: {len(new_data)} 条")
    return jsonl_path

def mix_jsonl(jsonl_a, jsonl_b, output_path):
    """1:1 混合两个 jsonl，轮流添加"""
    data_a = load_jsonl(jsonl_a)
    data_b = load_jsonl(jsonl_b)
    # 1:1 混合
    mixed = []
    for item_a, item_b in zip(data_a, data_b):
        mixed.append(item_a)
        mixed.append(item_b)
    with open(output_path, 'w') as f:
        for item in mixed:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  混合数据: {len(mixed)} 条 (A:{len(data_a)} + B:{len(data_b)})")
    return output_path

## 3. 生成混合数据

In [ ]:
print("=" * 60)
print("开始生成混合数据...")
print("=" * 60)

# Step 1: 生成纯噪音A数据
noise_a_only_path = generate_noise_a_only(
    raw_train,
    PREPARED_DIR,
    "train_noise_a_only.jsonl"
)

# Step 2: 生成纯噪音B数据
noise_b_only_path = generate_noise_b_only(
    raw_train,
    PREPARED_DIR,
    "train_noise_b_only.jsonl"
)

# Step 3: 原始数据 jsonl
raw_jsonl = os.path.join(PREPARED_DIR, "train_raw.jsonl")
with open(raw_jsonl, 'w') as f:
    for item in raw_train:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')
print(f"  原始数据: {len(raw_train)} 条")

# Step 4: 混合A = 原始数据 + 噪音A数据（1:1）
mixed_a_path = os.path.join(PREPARED_DIR, "train_mixed_a.jsonl")
mix_jsonl(raw_jsonl, noise_a_only_path, mixed_a_path)

# Step 5: 混合B = 原始数据 + 噪音B数据（1:1）
mixed_b_path = os.path.join(PREPARED_DIR, "train_mixed_b.jsonl")
mix_jsonl(raw_jsonl, noise_b_only_path, mixed_b_path)

print("\n数据生成完成!")
print(f"  混合A (原始+A): {mixed_a_path}")
print(f"  混合B (原始+B): {mixed_b_path}")

## 4. 训练

In [ ]:
import subprocess
import sys

def train_sensevoice_lora(train_jsonl, output_dir, exp_name, port=29500):
    os.makedirs(output_dir, exist_ok=True)
    cmd = [
        "torchrun", f"--nproc_per_node=1", f"--rdzv_endpoint=localhost:{port}",
        "-m", "funasr.bin.train_ds",
        "++model=iic/SenseVoiceSmall",
        f"++train_data_set_list={train_jsonl}",
        f"++valid_data_set_list={VAL_JSONL}",
        "++dataset_conf.data_split_num=1",
        "++dataset_conf.batch_sampler=BatchSampler",
        "++dataset_conf.batch_size=6000",
        "++dataset_conf.sort_size=1024",
        "++dataset_conf.batch_type=token",
        "++dataset_conf.num_workers=4",
        "++train_conf.max_epoch=10",
        "++train_conf.log_interval=50",
        "++train_conf.validate_interval=2000",
        "++train_conf.save_checkpoint_interval=2000",
        "++train_conf.keep_nbest_models=5",
        "++train_conf.avg_nbest_model=3",
        "++train_conf.use_bf16=true",
        "++train_conf.grad_clip=5",
        "++lora_only=true",
        "++lora_bias=none",
        "++lora_rank=8",
        "++lora_alpha=16",
        "++lora_dropout=0.1",
        "++optim=adamw",
        "++optim_conf.lr=2e-4",
        "++optim_conf.weight_decay=0.0",
        f"++output_dir={output_dir}",
    ]
    print(f"\n{'='*60}")
    print(f"训练: {exp_name}")
    print(f"数据: {train_jsonl}")
    print(f"输出: {output_dir}")
    print(f"端口: {port}")
    print(f"{'='*60}")
    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()
    returncode = process.wait()
    if returncode == 0:
        print(f"✓ {exp_name} 训练完成!")
    else:
        print(f"✗ {exp_name} 训练失败! 返回码: {returncode}")
    return returncode == 0

# --- 混合A训练 ---
ckpt_a = "/mnt/output/sv_mixed_a/model.pt.best"
if os.path.exists(ckpt_a):
    print("\n[跳过] 混合A 已完成")
    training_results = {"混合A": ckpt_a}
else:
    training_results = {}
    success = train_sensevoice_lora(
        mixed_a_path,
        "/mnt/output/sv_mixed_a",
        "混合A (原始+合成噪音)",
        port=29502,
    )
    training_results["混合A"] = "/mnt/output/sv_mixed_a/model.pt.best" if success else None

# --- 混合B训练 ---
ckpt_b = "/mnt/output/sv_mixed_b/model.pt.best"
if os.path.exists(ckpt_b):
    print("\n[跳过] 混合B 已完成")
    training_results["混合B"] = ckpt_b
else:
    success = train_sensevoice_lora(
        mixed_b_path,
        "/mnt/output/sv_mixed_b",
        "混合B (原始+真实噪音)",
        port=29503,
    )
    training_results["混合B"] = "/mnt/output/sv_mixed_b/model.pt.best" if success else None

print("\n训练汇总:")
for name, ckpt in training_results.items():
    status = "✓" if ckpt and os.path.exists(ckpt) else "✗"
    print(f"  {status} {name}: {ckpt}")